# Meta Summarization for Large Internal Context

This notebook builds an iterative **summary-of-summaries** pipeline to compress very large `internal_context` into a compact, high-signal `notes` string for downstream `ClientSummarization`.

## Goal

1. Split large `internal_context` into manageable chunks.
2. Summarize each chunk with strict preservation rules.
3. Merge chunk summaries iteratively until under target size.
4. Produce a final compressed notes block ready for `ClientSummarization(notes=...)`.

In [6]:
import dspy
import os
from dotenv import load_dotenv
from openinference.instrumentation.dspy import DSPyInstrumentor
from langfuse import get_client


load_dotenv("/home/azureuser/cloudfiles/code/email-generation/.env.local", override=True)

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3000"

langfuse = get_client()
DSPyInstrumentor().instrument()

In [7]:
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_base = os.getenv("AZURE_LANGUAGE_ENDPOINT")
api_version = os.getenv("AZURE_API_VERSION")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

lm_4o_mini = dspy.LM(
    model=f"azure/{os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}",
    api_key=api_key,
    api_base=api_base,
    # api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.8, 
    model_type='responses'
)

lm_41_mini = dspy.LM(
    model=f"azure/gpt-4.1-mini",
    api_key=os.getenv("AZURE_OPENAI_4_1_KEY"),
    api_base=api_base,
    # api_version="2025-04-14",
    temperature=0.8, 
    model_type='responses'
)


In [8]:
class MetaChunkSummarization(dspy.Signature):
    """
    You compress one chunk of internal AR context without losing critical facts.

    Rules:
    - Keep only factual content from input. Do not invent details.
    - Preserve all high-value entities if present: dates, invoice numbers, dollar amounts, commitments, blockers, owner names/roles, payment status, and next-step actions.
    - Collapse repetition. Keep newest updates before older background.
    - If multiple notes say the same thing, keep one representative statement and mark it as repeated.
    - Keep chronology clear.
    - Use compact prose, no bullets, no headings.
    - Output must be <= 1200 characters.
    """

    chunk_context: str = dspy.InputField(desc='Raw chunk from internal_context')
    compressed_chunk: str = dspy.OutputField(desc='Compact factual summary of this chunk')


class MetaMergeSummarization(dspy.Signature):
    """
    Merge multiple chunk summaries into one tighter timeline summary.

    Rules:
    - Preserve material facts and chronology from all provided chunk summaries.
    - Deduplicate repeated events.
    - Prioritize recency, unresolved blockers, overdue AR, commitments, and actionable next steps.
    - Keep dates in this format when available: Jan 01,2025.
    - No fabrication, no headings, no bullet points.
    - Output must be <= 1600 characters.
    """

    chunk_summaries: str = dspy.InputField(desc='Concatenated chunk summaries from prior stage')
    merged_summary: str = dspy.OutputField(desc='Merged compressed timeline summary')


class FinalNotesCompression(dspy.Signature):
    """
    Produce final compressed notes text for downstream ClientSummarization(notes=...).

    Rules:
    - Keep only essential facts needed for attorney-ready AR summarization.
    - Include current AR risk signals, recent interactions, critical historical context, and explicit next-step ownership.
    - Avoid narrative filler and repeated details.
    - No headings/bullets.
    - Output should fit limited prompt context and target <= 2000 characters.
    """

    merged_context: str = dspy.InputField(desc='Merged context from iterative meta summarization')
    compressed_notes: str = dspy.OutputField(desc='Final compact notes text for ClientSummarization module')


chunk_summarizer = dspy.Predict(MetaChunkSummarization)
merge_summarizer = dspy.Predict(MetaMergeSummarization)
final_compressor = dspy.Predict(FinalNotesCompression)

In [9]:
def split_text_with_overlap(text: str, max_chars: int = 8000, overlap_chars: int = 400):
    text = (text or '').strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    step = max_chars - overlap_chars
    if step <= 0:
        step = max_chars

    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step

    return chunks


def group_items_by_char_budget(items, group_budget_chars=8000):
    groups = []
    current = []
    current_len = 0

    for item in items:
        item_len = len(item)
        projected = current_len + item_len + (3 if current else 0)
        if current and projected > group_budget_chars:
            groups.append(current)
            current = [item]
            current_len = item_len
        else:
            current.append(item)
            current_len = projected

    if current:
        groups.append(current)

    return groups

In [10]:
def meta_summarize_internal_context(
    internal_context: str,
    chunk_max_chars: int = 8000,
    overlap_chars: int = 400,
    merge_group_chars: int = 8000,
    target_final_chars: int = 2000,
    max_rounds: int = 4,
):
    """
    Iterative hierarchical summarization:
    raw context -> chunk summaries -> merged summaries -> final compressed notes
    """

    if not internal_context or not internal_context.strip():
        return {
            'compressed_notes': '',
            'rounds_used': 0,
            'intermediate': []
        }

    level_items = [internal_context.strip()]
    intermediate = []

    for round_idx in range(1, max_rounds + 1):
        expanded_chunks = []
        for item in level_items:
            expanded_chunks.extend(split_text_with_overlap(item, max_chars=chunk_max_chars, overlap_chars=overlap_chars))

        chunk_summaries = []
        for chunk in expanded_chunks:
            chunk_response = chunk_summarizer(chunk_context=chunk)
            chunk_summaries.append(chunk_response.compressed_chunk.strip())

        if len(chunk_summaries) == 1:
            merged_level = chunk_summaries
        else:
            grouped = group_items_by_char_budget(chunk_summaries, group_budget_chars=merge_group_chars)
            merged_level = []
            for group in grouped:
                merge_input = '\n\n'.join(group)
                merge_response = merge_summarizer(chunk_summaries=merge_input)
                merged_level.append(merge_response.merged_summary.strip())

        snapshot = {
            'round': round_idx,
            'input_items': len(level_items),
            'chunks_created': len(expanded_chunks),
            'summaries_created': len(chunk_summaries),
            'merged_items': len(merged_level),
            'max_item_chars': max(len(x) for x in merged_level) if merged_level else 0
        }
        intermediate.append(snapshot)

        level_items = merged_level
        if len(level_items) == 1 and len(level_items[0]) <= target_final_chars:
            break

    merged_context = '\n\n'.join(level_items)
    final_response = final_compressor(merged_context=merged_context)
    compressed_notes = final_response.compressed_notes.strip()

    return {
        'compressed_notes': compressed_notes,
        'rounds_used': len(intermediate),
        'intermediate': intermediate,
        'input_chars': len(internal_context),
        'output_chars': len(compressed_notes),
        'compression_ratio': round(len(compressed_notes) / max(len(internal_context), 1), 4)
    }

In [ ]:
# Example usage
# internal_context = formatted_data  # or any large notes/activity context string
# meta_result = meta_summarize_internal_context(internal_context)
# compressed_notes = meta_result['compressed_notes']
# meta_result

In [ ]:
# Handoff into existing ClientSummarization module from summarization.ipynb
# Ensure `summarizer` or `summarizer_cot` is already defined.
#
# response = summarizer_cot(
#     financial_data=invoice_data,
#     notes=compressed_notes
# )
# response.summary

## Recommended Prompt Strategy

- Stage 1 (chunk): maximize factual recall, compress local repetition.
- Stage 2 (merge): deduplicate cross-chunk overlap, preserve chronology.
- Stage 3 (final): produce compact notes optimized for `ClientSummarization` context limits.

This 3-stage setup is safer than one-pass summarization when notes are very long and highly repetitive.